# 3. Variable objetivo

`solar_aptittude_class` clasifica cada sitio en tres niveles de aptitud solar: **Baja, Media y Alta**. Es un problema de **clasificación multiclase ordinal**.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import *

set_style()
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
df = load_data()

In [ ]:
dist = df[TARGET].value_counts().reindex(CLASS_ORDER)
pct = 100 * dist / dist.sum()

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.barh(CLASS_ORDER, dist.values, height=0.55,
        color=[CLASS_COLORS[c] for c in CLASS_ORDER])
for y, (n, p) in enumerate(zip(dist.values, pct.values)):
    ax.text(n + 500, y, f"{n:,}  ({p:.2f} %)", va="center", color="#0b0b0b")
ax.set_xlim(0, dist.max() * 1.3)
ax.set_xlabel("Número de plantas")
ax.set_title("Distribución de la clase de aptitud solar")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

pd.DataFrame({"plantas": dist, "porcentaje": pct.round(2)})

La clase está **fuertemente desbalanceada**: 75.10 % Alta, 21.69 % Media y apenas **3.21 % Baja** (1 892 plantas). La razón entre la clase mayoritaria y la minoritaria es de unos 23 a 1.

In [ ]:
ratio = dist["Alta"] / dist["Baja"]
print(f"Razón de desbalance Alta/Baja: {ratio:.1f} : 1")
print(f"Exactitud de un clasificador que siempre predice 'Alta': {pct['Alta']:.2f} %")

## 3.1 Implicaciones para el modelado

1. **La exactitud engaña.** Un modelo que siempre responde "Alta" ya obtiene un 75.1 % de exactitud sin aprender nada. En la segunda entrega, el modelo base alcanzó 0.787 de exactitud con un recall de Baja de solo 0.57. Las métricas principales deben ser el **F1 macro**, el **recall por clase** y el **ROC AUC macro**.
2. **La clase minoritaria es la más importante.** Desde la perspectiva de la decisión de inversión, identificar sitios de aptitud Baja es lo que evita malas inversiones.
3. **La partición debe ser estratificada.** En el 20 % de prueba solo quedan unas 379 plantas Baja, y sin estratificar esa cifra variaría de una partición a otra.
4. **Hay que tratar el desbalance.** Son candidatos `class_weight='balanced'`, que en la segunda entrega llevó el recall de Baja a 0.97, SMOTE y ADASYN, aplicados **solo** dentro de cada pliegue de entrenamiento.

## 3.2 Distribución de la clase en la partición 80/20

Se verifica que la partición estratificada con semilla 42, la misma de la segunda entrega, conserva las proporciones:

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=SEED)
pd.DataFrame({
    "train (%)": (100 * train[TARGET].value_counts(normalize=True)).reindex(CLASS_ORDER).round(2),
    "test (%)": (100 * test[TARGET].value_counts(normalize=True)).reindex(CLASS_ORDER).round(2),
    "test (n)": test[TARGET].value_counts().reindex(CLASS_ORDER),
})